In [ ]:
# 1. Time Series Classification Part 1

In [ ]:
#b

In [35]:
import os

data_dir = "data/activity+recognition+system+based+on+multisensor+data+fusion+arem"

# 7 activity folders
activities = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]

train_files = []
test_files = []

for activity in activities:
    folder = os.path.join(data_dir, activity)
    files = sorted(os.listdir(folder))

    for f in files:
        if not f.endswith(".csv"):
            continue

        num = int(f.replace("dataset", "").replace(".csv", ""))

        if activity in ["bending1", "bending2"]:
            # For bending1 and bending2: dataset1 and dataset2 are test data
            if num in [1, 2]:
                test_files.append(os.path.join(folder, f))
            else:
                train_files.append(os.path.join(folder, f))
        else:
            # For all other activities: dataset1, dataset2, dataset3 are test data
            if num in [1, 2, 3]:
                test_files.append(os.path.join(folder, f))
            else:
                train_files.append(os.path.join(folder, f))

print("Train files:", len(train_files))
print("Test files:", len(test_files))

Train files: 69
Test files: 19


In [ ]:
#c

In [ ]:
## i

In [ ]:
# When people work with time series data, they usually extract some simple numbers from each series to describe it. Common ones include:
# Minimum and maximum 
# Mean and median 
# Standard deviation 
# First and third quartiles 
# Skewness and kurtosis 
# Autocorrelation and entropy 

In [ ]:
## ii

In [36]:
import numpy as np
import pandas as pd

# 7 time-domain features to extract
feature_names = ["min", "max", "mean", "median", "std", "q1", "q3"]

def extract_features(file_path):
    # Read the CSV file, skip the first 6 comment lines
    df = pd.read_csv(file_path, skiprows=6,
                     names=["time", "avg_rss12", "var_rss12", "avg_rss13",
                            "var_rss13", "avg_rss23", "var_rss23"])

    # 6 time series columns
    ts_cols = ["avg_rss12", "var_rss12", "avg_rss13",
               "var_rss13", "avg_rss23", "var_rss23"]

    features = []
    for col in ts_cols:
        # Get the values of this time series
        series = df[col].values
        # Extract the 7 time-domain features
        features.extend([
            np.min(series),
            np.max(series),
            np.mean(series),
            np.median(series),
            np.std(series),
            np.percentile(series, 25),
            np.percentile(series, 75)
        ])
    return features

all_features = []
all_labels = []

for f in train_files:
    # Extract features from each training file
    feats = extract_features(f)
    all_features.append(feats)
    # The label is the activity name (folder name)
    label = f.split("/")[-2]
    all_labels.append(label)

# Column names: e.g., min_1, max_1, ..., q3_6
cols = [f"{feat}_{i+1}" for i in range(6) for feat in feature_names]

# Build the feature matrix and label vector
X_train = pd.DataFrame(all_features, columns=cols)
y_train = pd.Series(all_labels)

print("Feature matrix shape:", X_train.shape)
print(X_train.head())

Feature matrix shape: (69, 42)
   min_1  max_1     mean_1  median_1     std_1   q1_1   q3_1  min_2  max_2  \
0  35.00  47.40  43.958580     44.33  1.556270  43.00  45.00    0.0   1.70   
1  33.00  47.75  42.169061     43.50  3.663099  39.10  45.00    0.0   3.00   
2  33.00  45.75  41.672526     41.75  2.240209  41.33  42.75    0.0   2.83   
3  37.00  48.00  43.459562     43.25  1.382424  42.50  45.00    0.0   1.58   
4  36.25  48.00  43.973236     44.50  1.615855  43.33  44.67    0.0   1.50   

     mean_2  ...     std_5    q1_5   q3_5  min_6  max_6    mean_6  median_6  \
0  0.425658  ...  1.983199  35.450  36.50    0.0   1.79  0.494322      0.43   
1  0.696514  ...  3.849292  30.415  36.33    0.0   2.18  0.612088      0.50   
2  0.536117  ...  2.411018  28.415  31.25    0.0   1.79  0.384092      0.43   
3  0.376159  ...  2.452281  22.250  24.00    0.0   5.26  0.679102      0.50   
4  0.413987  ...  3.285963  20.500  23.75    0.0   2.96  0.556472      0.49   

      std_6  q1_6  q3_6  

In [ ]:
## iii

In [38]:
import numpy as np

# Number of bootstrap samples
n_boot = 1000

# Lists to store the confidence interval bounds
ci_lower = []
ci_upper = []

for col in X_train.columns:
    # Get the values of this feature
    data_col = X_train[col].values
    
    # Store the standard deviation of each bootstrap sample
    boot_stds = []
    
    for _ in range(n_boot):
        # Sample with replacement
        sample = np.random.choice(data_col, size=len(data_col), replace=True)
        # Compute the standard deviation of this sample
        boot_stds.append(np.std(sample))
    
    # 90% confidence interval: 5th and 95th percentiles
    ci_lower.append(np.percentile(boot_stds, 5))
    ci_upper.append(np.percentile(boot_stds, 95))

boot_results = pd.DataFrame({
    "Feature": X_train.columns,
    "Std": X_train.std().values,
    "CI_lower": ci_lower,
    "CI_upper": ci_upper
})

print(boot_results)

     Feature       Std  CI_lower  CI_upper
0      min_1  8.794295  7.471124  9.935350
1      max_1  4.429182  3.262191  5.345330
2     mean_1  4.918830  4.307752  5.351085
3   median_1  4.956571  4.294088  5.488398
4      std_1  1.756600  1.530624  1.931694
5       q1_1  5.731453  5.128591  6.164605
6       q3_1  4.786738  3.846478  5.432040
7      min_2  0.000000  0.000000  0.000000
8      max_2  5.147841  4.600368  5.494941
9     mean_2  1.600357  1.402956  1.726214
10  median_2  1.437295  1.235641  1.570145
11     std_2  0.901492  0.802353  0.953480
12      q1_2  0.951265  0.822031  1.040184
13      q3_2  2.157439  1.907789  2.319024
14     min_3  3.053869  2.822302  3.205340
15     max_3  4.736140  3.953500  5.335210
16    mean_3  3.863359  3.165564  4.393903
17  median_3  3.842930  3.119615  4.367465
18     std_3  0.993052  0.769150  1.190609
19      q1_3  4.145562  3.456069  4.677335
20      q3_3  3.944586  3.212528  4.466452
21     min_4  0.000000  0.000000  0.000000
22     max_

In [39]:
## iv

In [40]:
# I selected min, mean, and max as the three most important time-domain features. 
# These three capture the range and central tendency of each time series, 
# and they are commonly used in time series classification.

In [41]:
# 2. IRS 3.7.4

In [42]:
## a

In [43]:
# The cubic regression will have a training RSS that is lower than or equal to the linear regression. 
# The reason is that the linear regression is a special case of the cubic regression: 
# if we set β2 = 0 and β3 = 0, the cubic model becomes the linear model. 
# Since the cubic model has more parameters, it is more flexible and can fit the training data at least as well as the linear model. 
# Therefore, the training RSS of the cubic regression cannot be higher than that of the linear regression.

In [44]:
## b

In [45]:
# When using test RSS instead of training RSS, the answer may be different. 
# If the true relationship is linear, the linear regression is the correct model, 
# so it should have a lower test RSS. The cubic regression has more parameters and may overfit the training data, 
# fitting noise that does not generalize to new data. 
# Therefore, the cubic regression may have a higher test RSS than the linear regression, even though its training RSS is lower or equal.

In [46]:
## c

In [47]:
# The cubic regression will have a training RSS that is lower than or equal to the linear regression. 
# The reason is the same as in (a): 
# the linear regression is a special case of the cubic regression (when β2 = 0 and β3 = 0), 
# so the cubic model has more flexibility and can fit the training data at least as well as the linear model. \
# Moreover, since the true relationship is not linear, the extra terms X^2 and X^3 in the cubic model 
# will actually help capture the non-linear pattern, so the cubic regression's training RSS will likely 
# be noticeably lower than the linear regression's training RSS.

In [48]:
## d

In [49]:
# When using test RSS instead of training RSS, the answer depends on how far the true relationship is from linear. 
# If the true relationship is noticeably non-linear, the cubic regression will have a lower test RSS because 
# it can capture the non-linear pattern, while the linear regression will underfit. 
# However, if the true relationship is only slightly non-linear, the cubic regression may overfit the 
# training data and have a higher test RSS than the linear regression. 
# Since we do not know how far the relationship is from linear, 
# there is not enough information to tell which model will have a lower test RSS.